<a href="https://colab.research.google.com/github/ivashenceva220495-beep/My_first_repository/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22_%D0%94%D0%97_ML_%D0%A1%D0%B0%D0%BD%D0%B4%D1%8B%D0%B1%D0%B0%D0%B5%D0%B2%D0%B0_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Предсказываем стоимость мед страховки



## Загужаем необходимые библиотеки

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## Загружаем данные

Собраны данные:
- age: возраст
- sex: пол
- bmi: индекс массы тела
- children: количество детей, охваченных медицинским страхованием / количество иждивенцев
- smoker: курение
- region: регион (northeast, southeast, southwest, northwest).
- charges: индивидуальные медицинские расходы (его и хотим предсказать)

In [ ]:
# Загрузите данные из файла insurance.csv в переменную df
df = pd.read_csv('insurance.csv' , sep=',')


In [ ]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Смотрим статистику, что нет пропусков и отсуствующих значений

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


## Преобразуем строковые данные

In [ ]:
# Заменяем пол и курение на числа
df['sex']=df['sex'].map({'male':1, 'female':0})
df['smoker']=df['smoker'].map({'yes':1,'no':0})

In [ ]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,southwest,16884.92400
1,18,1,33.770,1,0,southeast,1725.55230
2,28,1,33.000,3,0,southeast,4449.46200
3,33,1,22.705,0,0,northwest,21984.47061
4,32,1,28.880,0,0,northwest,3866.85520


In [ ]:
# Заменяем регион на набор отдельных колонок (is_southwest, is_southeast и тп)
df = pd.get_dummies(df, columns=['region'])
df.head()

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,0,1
1,18,1,33.770,1,0,1725.55230,0,0,1,0
2,28,1,33.000,3,0,4449.46200,0,0,1,0
3,33,1,22.705,0,0,21984.47061,0,1,0,0
4,32,1,28.880,0,0,3866.85520,0,1,0,0


In [ ]:
df.head()

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,0,1
1,18,1,33.770,1,0,1725.55230,0,0,1,0
2,28,1,33.000,3,0,4449.46200,0,0,1,0
3,33,1,22.705,0,0,21984.47061,0,1,0,0
4,32,1,28.880,0,0,3866.85520,0,1,0,0


## Формируем признаки и целевую переменную

In [ ]:
## Сформируйте признаки и целевую переменную
df.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'charges',
       'region_northeast', 'region_northwest', 'region_southeast',
       'region_southwest'],
      dtype='object')

In [ ]:
X = df [['age', 'sex', 'bmi', 'children', 'smoker',
       'region_northeast', 'region_northwest', 'region_southeast',
       'region_southwest']]
y = df['charges']

In [ ]:
X

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,0,0,0,1
1,18,1,33.770,1,0,0,0,1,0
2,28,1,33.000,3,0,0,0,1,0
3,33,1,22.705,0,0,0,1,0,0
4,32,1,28.880,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...
1333,50,1,30.970,3,0,0,1,0,0
1334,18,0,31.920,0,0,1,0,0,0
1335,18,0,36.850,0,0,0,0,1,0
1336,21,0,25.800,0,0,0,0,0,1


In [ ]:
y

0       16884.92400
1        1725.55230
2        4449.46200
3       21984.47061
4        3866.85520
           ...     
1333    10600.54830
1334     2205.98080
1335     1629.83350
1336     2007.94500
1337    29141.36030
Name: charges, Length: 1338, dtype: float64

## Разделяем данные на выборку для обучения/проверки

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
287,63,0,26.220,0,0,0,1,0,0
26,63,0,23.085,0,0,1,0,0,0
350,57,0,23.180,0,0,0,1,0,0
1173,38,1,29.260,2,0,0,1,0,0
761,23,1,35.200,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...
1265,64,1,23.760,0,1,0,0,1,0
393,49,1,31.350,1,0,1,0,0,0
232,19,0,17.800,0,0,0,0,0,1
391,19,0,37.430,0,0,0,1,0,0


In [ ]:
X_test

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
1099,25,0,33.990,1,0,0,0,1,0
65,19,0,28.900,0,0,0,0,0,1
673,41,0,31.020,0,0,0,0,1,0
1165,35,0,26.125,0,0,1,0,0,0
222,32,1,30.800,3,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...
498,44,0,23.980,2,0,0,0,1,0
384,44,1,22.135,2,0,1,0,0,0
271,50,1,34.200,2,1,0,0,0,1
543,54,0,47.410,0,1,0,0,1,0


In [ ]:
y_train

287     14256.19280
26      14451.83515
350     11830.60720
1173     6457.84340
761      2416.95500
           ...     
1265    26926.51440
393      9290.13950
232      1727.78500
391      2138.07070
1245     5615.36900
Name: charges, Length: 1070, dtype: float64

In [ ]:
y_test

1099     3227.12110
65       1743.21400
673      6185.32080
1165     5227.98875
222      5253.52400
           ...     
498      8211.10020
384      8302.53565
271     42856.83800
543     63770.42801
501      6837.36870
Name: charges, Length: 268, dtype: float64

## Создаем и обучаем модель линейной регресии

In [ ]:
# Создайте и обучите модель
lr = LinearRegression()

In [ ]:
lr.fit(X_train, y_train)

LinearRegression()

## Получаем предсказание и оцениваем качество

In [ ]:
# Получите предсказание
lr.predict(X_test)

array([ 5402.74010183,  1476.1891673 ,  7955.15534305,  5900.22598135,
        7101.0628338 ,  4457.35987332, 14115.12695941,  6076.85572052,
        8351.48677982, 38231.03479386,  2664.3970151 , 30725.01609872,
        8406.58382779,  3509.31603406,  7311.48169282, 10672.60382015,
        5790.67321517, -2187.27029676,  8758.87389091, 17790.76886733,
        6352.15489391, 32846.087425  ,  3062.75300824,  8208.27316941,
       15667.10536527, 12841.87751567, 29794.33944469, 16242.22846744,
        6634.6931633 ,  6812.00099407,  4801.79348621, 27347.66463619,
       10229.04217056,  2515.51650748, -2219.08926541, 14788.42107539,
       36075.65126117, 35508.61415048, 12639.01194816, 16328.15521625,
       29458.66958207, 16272.88854081, 40470.80701688, 38523.18402726,
        7055.11158813, 32106.50223912,  9366.18065543, 31231.55470352,
       28003.95080041,  9405.0757114 , 13126.66719579, 10249.6313396 ,
       13728.02680973,  3778.29336354, 31382.91034631,  4804.13222422,
      

In [ ]:
# Оцените качество, при помощи метода mean_squared_error для тестовой выборки
mean_squared_error(y_test, lr.predict(X_test))

44070589.578941904

In [ ]:
mean_squared_error(y_train, lr.predict(X_train))

34711228.34880411

## Делаем предсказание для одного человека

In [ ]:
# Заполняем данные по конкретному человеку
data = [{
    "age": 20,
    "sex": 1,
    "bmi": 30,
    "children": 2,
    "smoker": 1,
    "region_northeast": 0,
    "region_northwest": 0,
    "region_southeast": 1,
    "region_southwest": 0
}]

In [ ]:
df_person = pd.DataFrame(data)
df_person.head()

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
0,20,1,30,2,1,0,0,1,0


In [ ]:
lr.predict(df_person)

array([26557.65787028])